# 第五讲 课程代码案例：NLP 进阶与模型微调

**受众**：Python 进阶课学生（已学完第4讲 NLP 基础与 Prompt 工程）
**前置条件**：了解 Tokenization、大模型 API 调用、Python 基础语法
**学习目标**：
1. 理解 Transformer 自注意力机制的数学原理与计算流程
2. 掌握三种预训练架构（BERT / GPT / T5）的异同与适用场景
3. 学会用 Hugging Face Transformers 对 BERT 进行情感分类微调
4. 了解 LoRA 参数高效微调的原理与基本用法
5. 能用 Vibe Coding 工具（CodeBuddy / Speckit）加速微调代码编写

**本 notebook 结构**：
- Part 1：Transformer 架构精讲（自注意力手动实现）
- Part 2：预训练模型家族对比（BERT/GPT/T5）
- Part 3：Hugging Face 生态与 pipeline 快速体验
- Part 4：BERT 情感分类微调实战（hotel.csv）
- Part 5：LoRA 参数高效微调
- Part 6：Vibe Coding 协作建模演示
- 知识点速查表 + 工具速查表


## Part 1：Transformer 架构精讲

**受众**：零基础学生
**前置条件**：了解神经网络基本概念（权重、梯度、激活函数）
**学习目标**：理解自注意力 Q/K/V 的计算流程，能用 NumPy 手动实现单头自注意力

### 1.1 语言模型演进：从 N-gram 到 Transformer

语言模型的核心任务是**计算一个词序列出现的概率**。我们先看最简单的 N-gram 模型，再理解为什么需要 Transformer。


In [1]:
# N-gram 模型演示：Bigram 概率计算
import collections

corpus = "datawhale agent learns datawhale agent works"
tokens = corpus.split()
total_tokens = len(tokens)

# 计算 Bigram 频次
bigrams = zip(tokens, tokens[1:])
bigram_counts = collections.Counter(bigrams)

# P(agent | datawhale) = Count("datawhale agent") / Count("datawhale")
count_datawhale = tokens.count('datawhale')
count_datawhale_agent = bigram_counts[('datawhale', 'agent')]
p_agent_given_datawhale = count_datawhale_agent / count_datawhale

print(f"语料: {corpus}")
print(f"总词数: {total_tokens}")
print(f"P(agent | datawhale) = {count_datawhale_agent}/{count_datawhale} = {p_agent_given_datawhale:.3f}")
print()
print("N-gram 局限：")
print("1. 数据稀疏：'robot learns' 没见过 → 概率为 0")
print("2. 泛化差：不理解 agent 和 robot 语义相似")


语料: datawhale agent learns datawhale agent works
总词数: 6
P(agent | datawhale) = 2/2 = 1.000

N-gram 局限：
1. 数据稀疏：'robot learns' 没见过 → 概率为 0
2. 泛化差：不理解 agent 和 robot 语义相似


**N-gram 的两大致命缺陷**：
1. **数据稀疏性**：未在语料中出现过的词序列概率为 0
2. **泛化能力差**：把词当作孤立符号，无法理解语义相似性

**解决思路**：用连续向量表示词（词嵌入 Word Embedding），让语义相近的词在向量空间中距离也近。


### 1.2 词嵌入与语义空间

词嵌入的核心思想：把每个词映射为高维连续向量，语义相近的词向量也相近。

经典案例：`King - Man + Woman ≈ Queen`


In [2]:
import numpy as np

# 简化的二维词向量（实际中维度通常为 100-768）
embeddings = {
    "king":   np.array([0.9, 0.8]),
    "queen":  np.array([0.9, 0.2]),
    "man":    np.array([0.7, 0.9]),
    "woman":  np.array([0.7, 0.3])
}

def cosine_similarity(vec1, vec2):
    """余弦相似度：衡量两个向量的方向一致性"""
    dot_product = np.dot(vec1, vec2)
    norm_product = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    return dot_product / norm_product

# 语义类比运算：king - man + woman
result_vec = embeddings["king"] - embeddings["man"] + embeddings["woman"]
sim_to_queen = cosine_similarity(result_vec, embeddings["queen"])

print(f"king - man + woman = {result_vec}")
print(f"queen              = {embeddings['queen']}")
print(f"结果向量与 queen 的余弦相似度: {sim_to_queen:.4f}")
print()
print("结论：词向量能捕捉'性别''皇室'等抽象语义概念")


king - man + woman = [0.9 0.2]
queen              = [0.9 0.2]
结果向量与 queen 的余弦相似度: 1.0000

结论：词向量能捕捉'性别''皇室'等抽象语义概念


### 1.3 Transformer 的革命（2017）

2017 年 Google 论文《Attention Is All You Need》提出 Transformer，完全抛弃循环结构，**只用注意力机制**实现并行计算。

| 对比项 | RNN/LSTM | Transformer |
|--------|----------|-------------|
| 计算方式 | 顺序计算 | 并行计算 |
| 长距离依赖 | 弱（梯度消失） | 强（一步直达） |
| 训练速度 | 慢 | 快（GPU 友好） |

Transformer 的核心组件：
1. **自注意力机制**（Scaled Dot-Product Attention）
2. **多头注意力**（Multi-Head Attention）
3. **前馈神经网络**（FFN）
4. **残差连接 + 层归一化**（Add & Norm）
5. **位置编码**（Positional Encoding）


### 1.4 自注意力机制：Q/K/V 三角色

**直觉理解**：阅读 "The agent learns because **it** is intelligent" 时，读到 "it" 大脑会关注 "agent"——自注意力就是这种过程的数学建模。

**三个核心角色**：
- **Query (Q)**：当前词想找什么信息（"你的问题"）
- **Key (K)**：这个词能提供什么索引（"书名标签"）
- **Value (V)**：这个词实际携带的内容（"书的内容"）

**四步计算**：
1. 计算相关性得分：$QK^T$
2. 缩放：$\div \sqrt{d_k}$
3. Softmax 归一化：转为和为 1 的权重
4. 加权求和：权重 × V

**最终公式**：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$


In [3]:
# 用 NumPy 手动实现单头自注意力
import numpy as np

def softmax(x):
    """对矩阵最后一个维度做 softmax（含数值稳定处理）"""
    x = x - np.max(x, axis=-1, keepdims=True)  # 减最大值防止 exp 溢出
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """
    缩放点积注意力
    参数：
        Q: (seq_len, d_k) 查询矩阵
        K: (seq_len, d_k) 键矩阵
        V: (seq_len, d_v) 值矩阵
    返回：
        output: (seq_len, d_v) 注意力输出
        weights: (seq_len, seq_len) 注意力权重
    """
    d_k = K.shape[-1]

    # Step 1: 计算 Q 和 K^T 的点积 → 相关性得分
    scores = Q @ K.T  # shape (seq_len, seq_len)

    # Step 2: 除以 sqrt(d_k) 缩放，防止梯度过小
    scaled_scores = scores / np.sqrt(d_k)

    # Step 3: Softmax 归一化，得到注意力权重（每行和为 1）
    weights = softmax(scaled_scores)

    # Step 4: 用注意力权重对 V 加权求和
    output = weights @ V  # shape (seq_len, d_v)

    return output, weights

# === 测试：3 个词，每个词 4 维嵌入 ===
np.random.seed(42)
seq_len, d_model = 3, 4
X = np.random.randn(seq_len, d_model)

# 简化：令 Q = K = V = X（实际中 Q/K/V 由 X 乘以权重矩阵得到）
Q, K, V = X, X, X

output, weights = scaled_dot_product_attention(Q, K, V)

print("输入矩阵 X (3 个词, 每词 4 维):")
print(np.round(X, 4))
print("\n注意力权重矩阵（每行和为 1）:")
print(np.round(weights, 4))
print(f"\n行和验证: {weights.sum(axis=1).round(4)}")
print("\n注意力输出（每个词融合全局信息后的新表示）:")
print(np.round(output, 4))


输入矩阵 X (3 个词, 每词 4 维):
[[ 0.4967 -0.1383  0.6477  1.523 ]
 [-0.2342 -0.2341  1.5792  0.7674]
 [-0.4695  0.5426 -0.4634 -0.4657]]

注意力权重矩阵（每行和为 1）:
[[0.5702 0.3641 0.0657]
 [0.3424 0.589  0.0686]
 [0.1918 0.2132 0.595 ]]

行和验证: [1. 1. 1.]

注意力输出（每个词融合全局信息后的新表示）:
[[ 1.6720e-01 -1.2850e-01  9.1390e-01  1.1173e+00]
 [-1.0000e-04 -1.4800e-01  1.1201e+00  9.4150e-01]
 [-2.3400e-01  2.4640e-01  1.8520e-01  1.7860e-01]]


**结果分析**：
- 注意力权重矩阵每行和为 1，表示每个词对其他词的关注度分配
- 权重越高，代表两个词关联性越强
- 输出是所有词向量的加权平均，每个词都"看到了"全局信息

**为什么除以 $\sqrt{d_k}$？** 当 $d_k$ 较大时，点积结果方差大，Softmax 后会落到梯度极小的区域（饱和区），缩放后让分布更平滑，梯度更稳定。


### 1.5 多头注意力：多角度并行关注

单头注意力只能学一种关联。多头思想：把 Q/K/V 在维度上切分成 h 份，每份独立做一次注意力，最后拼接。

```
原始 d_model=768, num_heads=12
每个头维度 = 768/12 = 64
12 个头各自关注不同关系（指代、时态、从属...）→ 拼接回 768 维
```

下面用 PyTorch 实现一个完整的多头注意力模块。


In [4]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    """多头注意力机制"""
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Q/K/V 和输出的线性变换层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # 1. 注意力得分 QK^T / sqrt(d_k)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # 2. 应用掩码（如有）
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        # 3. Softmax 归一化
        attn_probs = torch.softmax(attn_scores, dim=-1)
        # 4. 加权求和
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        # (batch, seq, d_model) → (batch, num_heads, seq, d_k)
        batch_size, seq_length, _ = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        # (batch, num_heads, seq, d_k) → (batch, seq, d_model)
        batch_size, _, seq_length, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        # 1. 线性变换 + 切分多头
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        # 2. 各头独立做注意力
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        # 3. 拼接 + 输出线性变换
        output = self.W_o(self.combine_heads(attn_output))
        return output

# 测试
mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 5, 64)  # batch=2, seq_len=5, d_model=64
output = mha(x, x, x)
print(f"输入 shape: {x.shape}")
print(f"输出 shape: {output.shape}")
print(f"参数量: {sum(p.numel() for p in mha.parameters())}")


输入 shape: torch.Size([2, 5, 64])
输出 shape: torch.Size([2, 5, 64])
参数量: 16640


## Part 2：预训练模型家族对比

**学习目标**：理解 BERT / GPT / T5 三种架构的设计逻辑与适用场景

### 2.1 三种架构总览

| 架构 | 代表模型 | 注意力方向 | 训练目标 | 擅长任务 |
|------|--------|-----------|---------|---------|
| **Encoder-Only** | BERT | 双向 | MLM（遮盖预测） | 理解类：分类、NER、问答 |
| **Decoder-Only** | GPT | 单向（因果） | CLM（预测下一个） | 生成类：对话、写作、代码 |
| **Encoder-Decoder** | T5 | 编码双向+解码因果 | Seq2Seq | 翻译、摘要、文本到文本 |


In [5]:
# 三种架构的注意力模式示意
import numpy as np

seq_len = 5

# BERT（Encoder-Only）：双向，每个位置看到所有位置
bert_mask = np.ones((seq_len, seq_len), dtype=int)

# GPT（Decoder-Only）：因果掩码，每个位置只能看前文
gpt_mask = np.tril(np.ones((seq_len, seq_len), dtype=int))

# T5（Encoder-Decoder）：Encoder 双向，Decoder 因果
t5_enc_mask = np.ones((seq_len, seq_len), dtype=int)
t5_dec_mask = np.tril(np.ones((seq_len, seq_len), dtype=int))

print("BERT 注意力掩码（全双向，1=可见）:")
print(bert_mask)
print("\nGPT 注意力掩码（因果，下三角）:")
print(gpt_mask)
print("\nT5 Encoder 掩码（双向）:")
print(t5_enc_mask)
print("\nT5 Decoder 掩码（因果）:")
print(t5_dec_mask)


BERT 注意力掩码（全双向，1=可见）:
[[1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]]

GPT 注意力掩码（因果，下三角）:
[[1 0 0 0 0]
 [1 1 0 0 0]
 [1 1 1 0 0]
 [1 1 1 1 0]
 [1 1 1 1 1]]

T5 Encoder 掩码（双向）:
[[1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]]

T5 Decoder 掩码（因果）:
[[1 0 0 0 0]
 [1 1 0 0 0]
 [1 1 1 0 0]
 [1 1 1 1 0]
 [1 1 1 1 1]]


### 2.2 BERT 的 MLM 训练目标

**Masked Language Model (MLM)**：随机遮盖 15% 的 token，让模型预测被遮盖的词。

```
原始：我爱北京天安门
遮盖：我 [MASK] 北京 [MASK] 门
目标：预测 [MASK] = "爱" 和 "天安"
```

BERT 的优势：**双向看到前后文**，更适合理解类任务（分类、NER、问答）。

### 2.3 GPT 的 CLM 训练目标

**Causal Language Model (CLM)**：逐词预测下一个词。

```
输入：[我, 爱, 北]
目标：[爱, 北, 京]
```

GPT 的优势：**自回归生成**，更适合生成类任务（对话、写作、代码）。


## Part 3：Hugging Face 生态与 pipeline 快速体验

**学习目标**：掌握 Hugging Face 核心 API，能用 pipeline 一行代码完成 NLP 任务

### 3.1 核心 API 速览

```python
from transformers import (
    AutoTokenizer,                        # 通用 Tokenizer
    AutoModel,                            # 通用模型（输出隐藏状态）
    AutoModelForSequenceClassification,   # 文本分类模型
    BertTokenizer,                        # BERT 专用
    pipeline                              # 一行代码完成任务
)
```


In [1]:
# pipeline 一行代码体验预训练模型
# 注意：首次运行会下载模型，需要网络
from transformers import pipeline

# 情感分析（英文）
classifier = pipeline("sentiment-analysis")
result = classifier("I love this hotel!")
print("英文情感分析:", result)
# [{'label': 'POSITIVE', 'score': 0.9998}]


/root/miniconda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3905.70it/s]


英文情感分析: [{'label': 'POSITIVE', 'score': 0.9998816251754761}]


In [9]:
ner = pipeline(
    "token-classification",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple"
)

result = ner("Zhang San is a student at Beihang University and works at Tencent.")

for ent in result:
    print(ent["word"], "→", ent["entity_group"])

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3131.83it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Zhang San → PER
Beihang University → ORG
Tencent → ORG


### 3.2 Tokenizer 深入

Tokenizer 把文本转换为模型能处理的数字 ID 序列。BERT 的 Tokenizer 输出三个字段：
- `input_ids`：词元对应的 ID 序列
- `attention_mask`：1=真实 token，0=padding
- `token_type_ids`：句对任务中区分上下句


In [20]:
# Tokenizer 编码示例
from transformers import BertTokenizer

# 加载中文 BERT Tokenizer
# 离线使用：BertTokenizer.from_pretrained("./bert-chinese")
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

text = "酒店位置很好，服务态度也不错。"
encoding = tokenizer(
    text,
    padding="max_length",     # 填充到最大长度
    truncation=True,          # 超长截断
    max_length=32,            # 最大长度
    return_tensors="pt"       # 返回 PyTorch 张量
)

print(f"原文: {text}")
print(f"input_ids: {encoding['input_ids'][0][:15].tolist()}...")  # 前 15 个
print(f"attention_mask: {encoding['attention_mask'][0][:15].tolist()}...")
print(f"token_type_ids: {encoding['token_type_ids'][0][:15].tolist()}...")
print()

# 解码回文本
decoded = tokenizer.decode(encoding['input_ids'][0], skip_special_tokens=True)
print(f"解码回文本: {decoded}")

# 特殊 token 说明
print()
print("特殊 token:")
print(f"  [CLS] (开头): {tokenizer.cls_token_id}")
print(f"  [SEP] (结尾): {tokenizer.sep_token_id}")
print(f"  [PAD] (填充): {tokenizer.pad_token_id}")
print(f"  [MASK] (遮盖): {tokenizer.mask_token_id}")


原文: 酒店位置很好，服务态度也不错。
input_ids: [101, 6983, 2421, 855, 5390, 2523, 1962, 8024, 3302, 1218, 2578, 2428, 738, 679, 7231]...
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]...
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...

解码回文本: 酒 店 位 置 很 好 ， 服 务 态 度 也 不 错 。

特殊 token:
  [CLS] (开头): 101
  [SEP] (结尾): 102
  [PAD] (填充): 0
  [MASK] (遮盖): 103


### 4.6 Loss 曲线诊断

| 现象 | 诊断 | 对策 |
|------|------|------|
| 训练/验证 Loss 都高 | 欠拟合 | 增加 epoch / 学习率 / 模型容量 |
| 训练↓ 验证↓ | 正常收敛 | 继续训练或停止 |
| 训练↓ 验证↑ | 过拟合 | 早停 / 加 Dropout / 减小 epoch / 加数据 |
| Loss 震荡剧烈 | 学习率过大 | 减小学习率，加 warmup |
| Loss 突然变 NaN | 梯度爆炸 | 加梯度裁剪 `clip_grad_norm_` |


## Part 5：LoRA 参数高效微调

**学习目标**：理解 LoRA 低秩分解原理，能用 peft 库实现轻量微调

### 5.1 全参数微调的代价

BERT-base 有 1.1 亿参数，全参数微调的问题：
- **显存大**：参数 + 梯度 + 优化器状态 ≈ 4 倍参数量
- **存储贵**：每个任务一份完整模型副本（400MB）
- **易过拟合**：参数多、数据少时易记忆训练集

### 5.2 LoRA 核心原理

**核心假设**：微调时的权重更新 ΔW 是低秩的，可分解为两个小矩阵乘积：

$$W_{\text{new}} = W_{\text{pretrained}} + \Delta W = W_{\text{pretrained}} + AB$$

- $W$：原始权重 (d, d)，**冻结不动**
- $A$：(d, r)，可训练
- $B$：(r, d)，可训练
- $r$：低秩维度，通常 4-64，远小于 d

**参数量对比**（d=768, r=8）：
- 原始：768 × 768 = 589,824
- LoRA：768 × 8 + 8 × 768 = 12,288（降到 2%）

**与 PCA 的相似之处**：都用低维空间近似高维变化，假设本征维度低。


In [ ]:
# LoRA 参数量对比演示
import numpy as np

d = 768  # 原始维度
r = 8    # 低秩维度

# 全参数微调：更新整个 W
full_params = d * d
print(f"全参数微调: {d} × {d} = {full_params:,} 参数")

# LoRA：只训练 A 和 B
lora_params = d * r + r * d
print(f"LoRA (r={r}): {d}×{r} + {r}×{d} = {lora_params:,} 参数")
print(f"参数占比: {lora_params/full_params*100:.2f}%")
print(f"参数减少: {(1 - lora_params/full_params)*100:.2f}%")


### 5.3 LoRA 代码实战


In [ ]:
# LoRA 微调代码（需要安装 peft：pip install peft）
from peft import LoraConfig, get_peft_model, TaskType

def build_lora_model():
    """构建 LoRA 微调模型"""
    # 1. 加载预训练模型（同任务2）
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-chinese", num_labels=2
    )

    # 2. 配置 LoRA
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,            # 序列分类任务
        r=8,                                    # 低秩维度
        lora_alpha=32,                          # 缩放系数（实际缩放 = alpha/r）
        lora_dropout=0.1,                       # LoRA 层 dropout
        target_modules=["query", "value"]       # 对注意力的 Q 和 V 矩阵加 LoRA
    )

    # 3. 用 LoRA 包装模型
    model = get_peft_model(model, lora_config)

    # 4. 查看可训练参数
    model.print_trainable_parameters()
    # trainable params: 294,912 || all params: 102,272,257 || trainable%: 0.29%

    return model

print("build_lora_model 定义完成")
# lora_model = build_lora_model()


### 5.4 全参数 vs LoRA 对比

| 对比项 | 全参数微调 | LoRA 微调 |
|--------|-----------|----------|
| 可训练参数量 | 1.1 亿 (100%) | 29 万 (0.29%) |
| 训练显存（batch=8） | 约 4GB | 约 1.5GB |
| 模型存储（每任务） | 400MB | 1MB（只存 LoRA 权重） |
| 最终 F1（hotel.csv） | 约 0.92 | 约 0.90 |
| 多任务部署 | 每任务一份完整模型 | 共享主干 + 多个 LoRA 头 |

**结论**：
- 数据量大、追求极致精度 → 全参数微调
- 数据少、多任务、资源受限 → LoRA
- 工程实践中 LoRA 已成为默认选择


## Part 6：Vibe Coding 协作建模演示

**学习目标**：用 Spec-driven + CodeBuddy 加速微调代码编写

### 6.1 Speckit 描述微调任务

把微调需求写成结构化 Spec，让 CodeBuddy 生成完整代码框架。

**Spec 示例**（`spec.md`）：

```markdown
# 任务：BERT 酒店评论情感分类微调

## 数据
- 文件：hotel.csv
- 字段：Comment（文本）、Label（0/1）

## 模型
- bert-base-chinese（本地 ./bert-chinese）

## 训练配置
- batch_size=8, max_length=128, lr=2e-5, epochs=3

## 输出
- 训练 Loss 曲线
- 6 条测试样例的预测结果
- F1 指标
```

把 Spec + 数据文件 `@` 引用给 CodeBuddy，一键生成训练脚本。


### 6.2 Inline Edit 调参

**场景**：训练 Loss 不降，需要调超参数。

**操作**：
1. 选中 `lr=2e-5` → `Cmd+K`（Mac）/ `Ctrl+K`（Win）
2. 输入："把学习率改成 5e-5，并加 warmup（前 10% 步数线性升温）"
3. 查看 Diff → Accept

```python
# 修改前
optimizer = AdamW(model.parameters(), lr=2e-5)

# 修改后（CodeBuddy 生成）
from transformers import get_linear_schedule_with_warmup
optimizer = AdamW(model.parameters(), lr=5e-5)
total_steps = len(train_loader) * 3
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)
```

### 6.3 斜杠指令调试

| 指令 | 用途 |
|------|------|
| `/fix` | 训练报错时一键定位问题 |
| `/explain` | 选中源码逐行解释（如 `BertForSequenceClassification`） |
| `/cr` | 审查微调代码的潜在问题 |
| `/tests` | 生成单元测试 |


## 知识点速查表

### Transformer 架构

| 组件 | 作用 | 关键公式 |
|------|------|---------|
| 自注意力 | 让每个词看到全局 | $\text{softmax}(QK^T/\sqrt{d_k})V$ |
| 多头注意力 | 多角度并行关注 | 切分 h 份各自注意力 → 拼接 |
| 前馈网络 (FFN) | 逐位置特征提取 | $\max(0, xW_1+b_1)W_2+b_2$ |
| 残差连接 | 防梯度消失 | $x + \text{Sublayer}(x)$ |
| 层归一化 | 稳定训练 | 均值 0、方差 1 |
| 位置编码 | 注入位置信息 | sin/cos 函数 |

### 预训练模型

| 架构 | 代表 | 注意力 | 训练目标 | 适用任务 |
|------|------|--------|---------|---------|
| Encoder-Only | BERT | 双向 | MLM | 分类、NER、问答 |
| Decoder-Only | GPT | 因果 | CLM | 生成、对话 |
| Encoder-Decoder | T5 | 编码双向+解码因果 | Seq2Seq | 翻译、摘要 |

### 微调关键概念

| 概念 | 说明 |
|------|------|
| 预训练 | 海量无标注数据自监督学习通用语言能力 |
| 微调 | 少量带标注数据监督学习特定任务能力 |
| MLM | 随机遮盖 15% token 预测，BERT 用 |
| CLM | 逐词预测下一个，GPT 用 |
| 全参数微调 | 更新所有参数，效果好但代价高 |
| LoRA | 冻结主干，只训练低秩矩阵 A、B，参数量 0.1%-1% |
| 学习率 | 微调通常 2e-5，过大导致灾难性遗忘 |
| warmup | 学习率从 0 线性升到 lr，稳定训练初期 |

## 工具速查表

| 操作 | 代码 |
|------|------|
| 加载 Tokenizer | `BertTokenizer.from_pretrained("bert-base-chinese")` |
| 加载分类模型 | `BertForSequenceClassification.from_pretrained(name, num_labels=2)` |
| 文本编码 | `tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="pt")` |
| 一行推理 | `pipeline("sentiment-analysis", model="bert-base-chinese")` |
| 前向传播 | `outputs = model(input_ids=..., attention_mask=..., labels=...)` |
| 优化器 | `AdamW(model.parameters(), lr=2e-5)` |
| LoRA 配置 | `LoraConfig(task_type=TaskType.SEQ_CLS, r=8, target_modules=["query","value"])` |
| LoRA 包装 | `model = get_peft_model(model, lora_config)` |

## Vibe Coding 工具速查

| 工具 | 用途 | 何时用 |
|------|------|--------|
| Speckit | 描述任务规格 | 开始一个新微调任务时 |
| CodeBuddy Craft | 生成代码框架 | 从 Spec 生成训练脚本 |
| Inline Edit (Cmd+K) | 行内编辑 | 调超参数、改小细节 |
| Diff 审查 | 查看修改前后对比 | 每次 AI 修改后先看再 Accept |
| `/fix` | 修复错误 | 训练报错时 |
| `/explain` | 解释代码 | 看不懂源码时 |
| `/cr` | 代码审查 | 提交前检查 |
| Rules | 定义项目风格 | 配置编码规范 |
